# Physics API Stress Test - 100 Type 2 Questions

This notebook tests the running `/predict` endpoint with 100 physics Type 2 questions. It sends one request per question, matching the organizer quick-check style. The goal is to measure answer correctness, unit correctness, latency, deterministic/semantic usage, and timeout risk.

Run this after the API notebook is already running and ngrok has printed the public `/predict` URL.

## 1. Configuration

Paste the active `/predict` URL below. Keep `POLISH=False` for stress testing answer/unit accuracy and latency. Turn polish on only after the solver is stable.

In [ ]:
import csv
import json
import math
import time
import urllib.request
import urllib.error
from pathlib import Path

import pandas as pd

# Paste your active public or local /predict endpoint here.
PREDICT_URL = 'https://YOUR-NGROK-URL.ngrok-free.dev/predict'

# For grading safety tests, keep polish off. Semantic parser can still run if the API enables it.
POLISH = False
DEBUG = True
REQUEST_TIMEOUT_SECONDS = 120
SLEEP_BETWEEN_REQUESTS = 0.10

OUTPUT_CSV = Path('/kaggle/working/physics_api_stress_100_results.csv')
OUTPUT_JSONL = Path('/kaggle/working/physics_api_stress_100_raw.jsonl')

def normalize_predict_url(url):
    url = str(url or '').strip()
    if not url:
        raise ValueError('PREDICT_URL is empty. Paste the active /predict endpoint.')
    url = url.rstrip('/')
    for suffix in ['/health', '/v1/models', '/solve', '/batch']:
        if url.endswith(suffix):
            url = url[: -len(suffix)]
            break
    if not url.endswith('/predict'):
        url = url + '/predict'
    return url

PREDICT_URL = normalize_predict_url(PREDICT_URL)

print('Predict URL:', PREDICT_URL)
print('Polish:', POLISH)

## 2. Load 100 Handwritten Stress Questions

The questions are stored in `physics_stress_100_handwritten.json`. This notebook does not generate test questions; it only loads the prepared JSON file and sends each query to the API.


In [ ]:
from pathlib import Path
import json

TEST_SET_FILENAME = 'physics_stress_100_handwritten.json'
SEARCH_ROOTS = [
    Path('/kaggle/input'),
    Path('/kaggle/working'),
    Path.cwd(),
    Path.cwd() / 'result' / 'test_sets',
    Path.cwd() / 'kaggle_api_package_v1' / 'result' / 'test_sets',
]

def find_test_set(filename=TEST_SET_FILENAME):
    candidates = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            candidates.append(direct)
        candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(f'Could not find {filename}. Upload the package dataset or place the JSON in /kaggle/working.')
    def score(path):
        text = str(path).lower()
        value = 0
        if 'kaggle_api_package_v1' in text:
            value += 20
        if 'test_sets' in text:
            value += 10
        if '/kaggle/input' in text.replace('\\', '/'):
            value += 5
        return value
    return sorted(set(candidates), key=score, reverse=True)[0]

TEST_SET_PATH = find_test_set()
CASES = json.loads(TEST_SET_PATH.read_text(encoding='utf-8'))
assert isinstance(CASES, list), 'Test set must be a JSON list.'
assert len(CASES) == 100, f'Expected 100 cases, found {len(CASES)}.'
required = {'query_id', 'topic', 'query', 'expected_answer', 'expected_unit'}
for row in CASES:
    missing = required - set(row)
    if missing:
        raise ValueError(f"Missing keys {missing} in row {row.get('query_id')}")
    row.setdefault('rel_tol', 0.035)
    row.setdefault('abs_tol', 1e-9)

print('Loaded test set:', TEST_SET_PATH)
print('Total cases:', len(CASES))
print('First case:', CASES[0]['query_id'], '-', CASES[0]['query'])


## 3. Request And Scoring Helpers

The checker compares numeric values after converting common units. It also records method, confidence, semantic status, polish status, and latency.

In [ ]:
UNIT_SCALE = {
    '': 1.0, '-': 1.0,
    'A': 1.0, 'mA': 1e-3, 'uA': 1e-6,
    'V': 1.0, 'mV': 1e-3, 'kV': 1e3,
    'ohm': 1.0, '?': 1.0,
    'W': 1.0,
    'J': 1.0, 'mJ': 1e-3, 'uJ': 1e-6, 'nJ': 1e-9,
    'F': 1.0, 'uF': 1e-6, 'nF': 1e-9, 'pF': 1e-12,
    'H': 1.0, 'mH': 1e-3, 'uH': 1e-6,
    'C': 1.0, 'uC': 1e-6, 'nC': 1e-9, 'pC': 1e-12,
    'Hz': 1.0, 'rad/s': 1.0,
    'T': 1.0, 'mT': 1e-3, 'uT': 1e-6,
    'Wb': 1.0,
    'V/m': 1.0, 'N/C': 1.0,
    'N': 1.0,
    '%': 1.0,
}

def canonical_unit(unit):
    text = str(unit or '').strip()
    text = text.replace('?', 'ohm').replace('Ohm', 'ohm').replace('ohms', 'ohm')
    text = text.replace('microfarad', 'uF').replace('microcoulomb', 'uC')
    return text

def parse_number(value):
    text = str(value or '').strip().replace(',', '.')
    text = text.replace('?', 'x').replace('*10^', 'e').replace('x10^', 'e')
    import re
    text = re.sub(r'([+-]?\d+(?:\.\d+)?)\s*(?:x|\*)\s*10\s*\^?\s*([+-]?\d+)', r'\1e\2', text, flags=re.I)
    m = re.search(r'[+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?', text, flags=re.I)
    return float(m.group(0)) if m else None

def unit_scale(unit):
    return UNIT_SCALE.get(canonical_unit(unit), 1.0)

def is_correct(answer, unit, expected_answer, expected_unit, rel_tol=0.035, abs_tol=1e-9):
    got = parse_number(answer)
    exp = float(expected_answer)
    if got is None:
        return False
    got_si = got * unit_scale(unit)
    exp_si = exp * unit_scale(expected_unit)
    return math.isclose(got_si, exp_si, rel_tol=float(rel_tol), abs_tol=float(abs_tol))

def extract_result_item(parsed):
    # Correct competition endpoint: list with one object.
    if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
        return parsed[0], ''
    # Helpful fallback if user accidentally points at /solve instead of /predict.
    if isinstance(parsed, dict) and any(k in parsed for k in ['answer', 'unit', 'explanation']):
        return parsed, 'non_list_response_but_contains_answer'
    return {}, f'unexpected_response_shape:{type(parsed).__name__}'

def post_one(case):
    payload = {
        'query_id': case['query_id'],
        'type': 'type2',
        'query': case['query'],
        'premises': [],
        'options': [],
        'debug': DEBUG,
        'polish': POLISH,
    }
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        PREDICT_URL,
        data=data,
        headers={
            'Content-Type': 'application/json',
            'Accept': 'application/json',
            'ngrok-skip-browser-warning': 'true',
        },
        method='POST',
    )
    started = time.time()
    try:
        with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_SECONDS) as resp:
            raw = resp.read().decode('utf-8', errors='replace')
            status = resp.status
        elapsed_ms = round((time.time() - started) * 1000, 2)
        try:
            parsed = json.loads(raw)
        except Exception as exc:
            raise ValueError(f'API did not return JSON: {raw[:500]}') from exc
        item, shape_warning = extract_result_item(parsed)
        debug = item.get('debug', {}) if isinstance(item, dict) else {}
        ok = is_correct(
            item.get('answer', ''),
            item.get('unit', ''),
            case['expected_answer'],
            case['expected_unit'],
            case.get('rel_tol', 0.035),
            case.get('abs_tol', 1e-9),
        )
        error_text = shape_warning
        if not item.get('answer', '') and shape_warning:
            error_text = shape_warning + ' | raw=' + raw[:500].replace('\n', ' ')
        return {
            **case,
            'http_status': status,
            'answer': item.get('answer', ''),
            'unit': item.get('unit', ''),
            'correct': ok,
            'elapsed_ms': elapsed_ms,
            'method': debug.get('method', ''),
            'confidence': debug.get('confidence', ''),
            'topic_pred': debug.get('topic_pred', ''),
            'prefix_pred': debug.get('prefix_pred', ''),
            'semantic_status': debug.get('semantic_status', ''),
            'semantic_model': debug.get('semantic_model', ''),
            'semantic_repair_status': debug.get('semantic_repair_status', ''),
            'polish_status': debug.get('polish_status', ''),
            'error': error_text,
            'raw_response': raw[:2000],
            'explanation': item.get('explanation', ''),
        }
    except Exception as exc:
        elapsed_ms = round((time.time() - started) * 1000, 2)
        return {
            **case,
            'http_status': '',
            'answer': '',
            'unit': '',
            'correct': False,
            'elapsed_ms': elapsed_ms,
            'method': '',
            'confidence': '',
            'topic_pred': '',
            'prefix_pred': '',
            'semantic_status': '',
            'semantic_model': '',
            'semantic_repair_status': '',
            'polish_status': '',
            'error': repr(exc),
            'raw_response': '',
            'explanation': '',
        }


## 4. Run 100 Requests

This cell sends requests sequentially. It writes partial results after every request, so you still keep the CSV if the run is interrupted.

In [ ]:
rows = []
OUTPUT_JSONL.unlink(missing_ok=True)

for idx, case in enumerate(CASES, 1):
    row = post_one(case)
    rows.append(row)
    with OUTPUT_JSONL.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    marker = 'OK' if row['correct'] else 'FAIL'
    err = f" error={row['error'][:160]}" if row.get('error') else ''
    print(f"{idx:03d}/100 {marker} {row['query_id']} got={row['answer']} {row['unit']} expected={row['expected_answer']:.6g} {row['expected_unit']} time={row['elapsed_ms']}ms method={row['method']} semantic={row['semantic_status']}{err}")
    time.sleep(SLEEP_BETWEEN_REQUESTS)

results = pd.DataFrame(rows)
print('\nSaved:', OUTPUT_CSV)
print('Saved:', OUTPUT_JSONL)
results.head()


## 5. Summary And Failure Analysis

Use these tables to decide what to fix next: deterministic routing, semantic parser timeout, unit normalization, or canonical solver coverage.

In [ ]:
results = pd.read_csv(OUTPUT_CSV)
print('Total:', len(results))
print('Correct:', int(results['correct'].sum()))
print('Accuracy:', round(float(results['correct'].mean()), 4))
print('Median latency ms:', round(float(results['elapsed_ms'].median()), 2))
print('P95 latency ms:', round(float(results['elapsed_ms'].quantile(0.95)), 2))
print('Max latency ms:', round(float(results['elapsed_ms'].max()), 2))

print('\nBy topic:')
display(results.groupby('topic').agg(
    n=('query_id', 'count'),
    correct=('correct', 'sum'),
    accuracy=('correct', 'mean'),
    median_ms=('elapsed_ms', 'median'),
    max_ms=('elapsed_ms', 'max'),
).reset_index())

print('\nBy method:')
display(results.groupby('method').agg(
    n=('query_id', 'count'),
    correct=('correct', 'sum'),
    accuracy=('correct', 'mean'),
    median_ms=('elapsed_ms', 'median'),
    max_ms=('elapsed_ms', 'max'),
).reset_index().sort_values(['accuracy', 'n'], ascending=[True, False]))

print('\nSemantic statuses:')
display(results.groupby('semantic_status', dropna=False).agg(
    n=('query_id', 'count'),
    correct=('correct', 'sum'),
    median_ms=('elapsed_ms', 'median'),
).reset_index())


## 6. Inspect Failed Cases

Send the saved CSV back to Codex when failures appear. The most useful columns are query, expected answer/unit, API answer/unit, method, semantic status, latency, and explanation.

In [ ]:
failed = results[~results['correct'].astype(bool)].copy()
print('Failed:', len(failed))
cols = [
    'query_id', 'topic', 'expected_answer', 'expected_unit', 'answer', 'unit',
    'elapsed_ms', 'method', 'confidence', 'semantic_status', 'semantic_repair_status',
    'polish_status', 'error', 'query', 'explanation'
]
display(failed[cols].head(50))

# Save failures separately for easier upload/debugging.
FAIL_CSV = Path('/kaggle/working/physics_api_stress_100_failures.csv')
failed.to_csv(FAIL_CSV, index=False, encoding='utf-8-sig')
print('Saved failures:', FAIL_CSV)
